In [1]:
import pandas as pd
import os
import numpy as np

In [2]:

raw_video_dir = '/mnt/camera_nas'

In [3]:
full_data = '/media/dherrera/ElephantsWD/elephants/reid_gt_cleaned_data//train_val_split/full_reid_data.csv'
train_csv = '/media/dherrera/ElephantsWD/elephants/reid_gt_cleaned_data/train_val_split/train.csv'
val_csv = '/media/dherrera/ElephantsWD/elephants/reid_gt_cleaned_data/train_val_split/val.csv'


df_full = pd.read_csv(full_data)
df_train = pd.read_csv(train_csv)
df_val = pd.read_csv(val_csv)

In [4]:
df_full.head()

,file,id,name,cam_id,date,timestamp
0,2025_07/01_Chandra/zag_elp_cam_019_20250702_06...,1,Chandra,cam_019,2025-07-02,2025-07-02T06:35:10
1,2025_07/01_Chandra/zag_elp_cam_019_20250702_06...,1,Chandra,cam_019,2025-07-02,2025-07-02T06:35:12
2,2025_07/01_Chandra/zag_elp_cam_017_20250707_06...,1,Chandra,cam_017,2025-07-07,2025-07-07T06:36:28
3,2025_07/01_Chandra/zag_elp_cam_017_20250707_06...,1,Chandra,cam_017,2025-07-07,2025-07-07T06:36:30
4,2025_07/01_Chandra/zag_elp_cam_017_20250707_06...,1,Chandra,cam_017,2025-07-07,2025-07-07T06:34:32


In [5]:
df_full['file'][0]

'2025_07/01_Chandra/zag_elp_cam_019_20250702_063510_t2_40.jpg'

In [6]:
def time2ampm(time_str):
    """
    Convert time in 'HHMMSS' format to 'AM/PM' format.
    e.g. '063510' -> 'AM'
    """
    hour = int(time_str[0:2])
    ampm = 'AM' if hour < 12 else 'PM'
    return ampm


def extract_metadata_from_filepath(filepath):
    """
    e.g. '2025_07/01_Chandra/zag_elp_cam_019_20250702_063510_t2_40.jpg' -> ('019', '20250702', '063510')
    """
    parts = filepath.split('/')
    filename = parts[-1]
    camera_id = filename.split('_')[3]
    date = filename.split('_')[4]
    time = filename.split('_')[5]
    ampm = time2ampm(time)
    return camera_id, date, time, ampm


In [7]:
def match_time_to_video_file(raw_video_dir, camera_id, date, time, ampm):
    """
    Match the given time string to the corresponding video file in the directory.
    Arguments:
        raw_video_dir: str, base directory where raw videos are stored
        camera_id: str, camera ID extracted from the filename
        date: str, date extracted from the filename in 'YYYYMMDD' format
        time: str, time extracted from the filename in 'HHMMSS' format
        ampm: str, 'AM' or 'PM' based on the time
        find_opposite: bool, if True, find video from opposite camera (016<->019, 017<->018)
    Returns:
        matched_video_file: str or None, path to the matched video file or None if not found
    """

    video_dir = f'{raw_video_dir}/ZAG-ELP-CAM-{camera_id}/{date}{ampm}'

    video_files = [f for f in os.listdir(video_dir) if f.startswith(f'ZAG-ELP-CAM-{camera_id}-{date}')]

    for video_file in video_files:
        video_starting_time = video_file.split('-')[5]  # Extract time part from filename
        
        # check if the given time is within the video duration (~2 hours duration for each video)
        video_start_hour = int(video_starting_time[0:2])
        video_start_minute = int(video_starting_time[2:4])
        video_start_second = int(video_starting_time[4:6])
        given_hour = int(time[0:2])
        given_minute = int(time[2:4])
        given_second = int(time[4:6])
        time_diff = (given_hour - video_start_hour) * 3600 + (given_minute - video_start_minute) * 60 + (given_second - video_start_second)
        if 0 <= time_diff <= 2 * 3600:
            matched_video_file = os.path.join(video_dir, video_file)
            # print("Matched video file:", matched_video_file)
            return matched_video_file
    
    return None


CAMERA_PARIS = {
    '016': '019',
    '019': '016',
    '017': '018',
    '018': '017',
}


In [9]:
file = df_train['file'][0]

camera_id, date, time, ampm = extract_metadata_from_filepath(file)
matched_video_file = match_time_to_video_file(raw_video_dir, camera_id, date, time, ampm)

# find opposite camera video
opposite_camera_id = CAMERA_PARIS[camera_id]
opposite_matched_video_file = match_time_to_video_file(raw_video_dir, opposite_camera_id, date, time, ampm)

print("Original camera video file:", matched_video_file)
print("Opposite camera video file:", opposite_matched_video_file)


### copy opposite_matched_video_file to /home/mu/Desktop/gt_videos/train_matched
opposite_matched_dir = '/home/mu/Desktop/gt_videos/train_matched'
os.makedirs(opposite_matched_dir, exist_ok=True)
opposite_matched_filename = os.path.basename(opposite_matched_video_file)
opposite_matched_dest = os.path.join(opposite_matched_dir, opposite_matched_filename)
os.system(f'cp "{opposite_matched_video_file}" "{opposite_matched_dest}"')


Original camera video file: /mnt/camera_nas/ZAG-ELP-CAM-019/20250702AM/ZAG-ELP-CAM-019-20250702-063508-1751430908043-7.mp4
Opposite camera video file: /mnt/camera_nas/ZAG-ELP-CAM-016/20250702AM/ZAG-ELP-CAM-016-20250702-063257-1751430777352-7.mp4


0

In [ ]:
df_train['video_file'] = df_train['file'].apply(lambda x: match_time_to_video_file(
    raw_video_dir, *extract_metadata_from_filepath(x)
))
df_val['video_file'] = df_val['file'].apply(lambda x: match_time_to_video_file(
    raw_video_dir, *extract_metadata_from_filepath(x)
))

df_train

In [22]:
df_train.to_csv('/home/mu/Desktop/gt_csv/train.csv', index=False)
df_val.to_csv('/home/mu/Desktop/gt_csv/val.csv', index=False)

In [ ]:
train_videos = df_train['video_file'].unique()
val_videos = df_val['video_file'].unique()



(array(['/mnt/camera_nas/ZAG-ELP-CAM-019/20250812AM/ZAG-ELP-CAM-019-20250812-065808-1754974688085-7.mp4',
        '/mnt/camera_nas/ZAG-ELP-CAM-019/20250820AM/ZAG-ELP-CAM-019-20250820-025815-1755651495059-7.mp4'],
       dtype=object),
 array(['/mnt/camera_nas/ZAG-ELP-CAM-019/20250704AM/ZAG-ELP-CAM-019-20250704-063518-1751603718720-7.mp4',
        '/mnt/camera_nas/ZAG-ELP-CAM-018/20250813AM/ZAG-ELP-CAM-018-20250813-065812-1755061092000-7.mp4'],
       dtype=object))

In [26]:
## random pick K videos from train and val to visualize (focus on PM videos)
np.random.seed(42)
K = 10
sample_train_videos = np.random.choice(
    [v for v in train_videos if 'PM' in v], size=K, replace=False
)
sample_val_videos = np.random.choice(
    [v for v in val_videos if 'PM' in v], size=K, replace=False
)

## copy these video files to local for visualization
import shutil
for video_file in sample_train_videos:
    dest_path = os.path.join('/home/mu/Desktop/gt_videos/train', os.path.basename(video_file))
    shutil.copy(video_file, dest_path)

for video_file in sample_val_videos:
    dest_path = os.path.join('/home/mu/Desktop/gt_videos/val', os.path.basename(video_file))
    shutil.copy(video_file, dest_path)

In [ ]:
video = '/home/mu/Desktop/gt_videos/train/ZAG-ELP-CAM-018-20250830-025815-1756515495749-7.mp4'